# Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/spinal-bone-feature-detection
!ls

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/.shortcut-targets-by-id/1oG2uzFRqi9wz763XVJclWvhPaLhUUFFX/spinal-bone-feature-detection
backup	     datasets	  kfolds     models.py	  results      utils.py
config.yaml  experiments  loader.py  __pycache__  trainers.py  weights


In [ ]:
!unzip -q ./datasets/ultrasound.zip -d /tmp/ultrasound
!pip install -q torchmetrics[detection]
# !pip install -q torch_geometric

In [ ]:
import torch
import torch.optim as optim
from torchvision.ops import distance_box_iou_loss

import gc
from models import MultiTaskModel
from trainers import MultiTaskModelTrainer
from loader import get_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits_path = 'kfolds/splits4.yaml'
best_model_path = 'weights/diou4.pth'
!rm -rf results
%load_ext tensorboard

# Training

In [ ]:
train_loader = get_loader('train', splits_path=splits_path, batch_size=32)
val_loader = get_loader('val', splits_path=splits_path, batch_size=32)
model = MultiTaskModel(criterion_box=distance_box_iou_loss, device=device)
optimizer = optim.AdamW(model.parameters(), lr=5e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-7)

[kfolds/splits4.yaml] Train dataset: 248 samples
[kfolds/splits4.yaml] Val dataset: 61 samples


In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()
trainer = MultiTaskModelTrainer(model, train_loader, val_loader, num_epochs=1000, optimizer=optimizer, best_model_path=best_model_path)
trainer.train()

# mAP Evaluation

In [ ]:
# %tensorboard --logdir results

In [ ]:
checkpoint = torch.load(best_model_path)
trainer.model.load_state_dict(checkpoint)
mAP_preds, mAP_targets, results = trainer.evaluate()
results